# BadBlueprint full scoring (gpt-oss-20b) on Colab

This notebook runs the full BadBlueprint scoring flow end-to-end using the local open-weight model and the vendored harness.

## A. Runtime / GPU check

In [ ]:
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Expect slow performance.")

print("
System info:")
print(subprocess.run(["/bin/bash", "-lc", "free -h"], capture_output=True, text=True).stdout)
print(subprocess.run(["/bin/bash", "-lc", "df -h /"], capture_output=True, text=True).stdout)

## B. Environment setup

In [ ]:
import sys
import subprocess

# Remove commonly conflicting packages if present (safe to ignore failures)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=False)

# Install required dependencies (minimal set)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git",
    "accelerate", "safetensors", "huggingface_hub", "uvicorn", "fastapi", "requests"
], check=True)

## C. Model download

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

model_id = "openai/gpt-oss-20b"
model_dir = Path("/content/models/gpt-oss-20b")
model_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")

snapshot_download(
    repo_id=model_id,
    local_dir=str(model_dir),
    local_dir_use_symlinks=False,
    token=hf_token,
)

files = list(model_dir.glob("**/*"))
print(f"Model downloaded to: {model_dir}")
print(f"Files found: {len(files)}")

## D. Start a local OpenAI-compatible endpoint

In [ ]:
import os
import signal
import subprocess
import time
import shutil

MODEL_ENDPOINT = "http://127.0.0.1:8000/v1"
MODEL_DIR = "/content/models/gpt-oss-20b"

server_cmd = None
if shutil.which("transformers"):
    server_cmd = [
        "transformers", "serve",
        "--model", MODEL_DIR,
        "--host", "127.0.0.1",
        "--port", "8000",
    ]

if server_cmd is None:
    raise RuntimeError("No supported transformers serve command found. Update transformers.")

if "_model_server_proc" in globals():
    if _model_server_proc.poll() is None:
        print("Server already running.")
    else:
        _model_server_proc = None

if "_model_server_proc" not in globals() or _model_server_proc is None:
    _model_server_proc = subprocess.Popen(
        server_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    print(f"Started server PID: {_model_server_proc.pid}")

In [ ]:
import time
import requests

MODEL_ENDPOINT = "http://127.0.0.1:8000/v1"

def wait_for_server(url, retries=20, delay=3):
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(f"{url}/models", timeout=5)
            if resp.status_code == 200:
                print("Server is healthy.")
                return resp.json()
        except Exception:
            pass
        print(f"Waiting for server... ({attempt}/{retries})")
        time.sleep(delay)
    raise RuntimeError("Server did not become healthy in time.")

models_payload = wait_for_server(MODEL_ENDPOINT)
print(models_payload)

MODEL_NAME = "openai/gpt-oss-20b"
try:
    model_ids = [m.get("id") for m in models_payload.get("data", []) if isinstance(m, dict)]
    if model_ids:
        MODEL_NAME = model_ids[0]
except Exception:
    pass

print(f"Using model name: {MODEL_NAME}")

## E. Clone repo and prepare submission bundle

In [ ]:
import os
import subprocess
from pathlib import Path

repo_name = "purple-vanguard-scenarios"
repo_path = Path("/content") / repo_name

if repo_path.exists():
    print(f"Using existing repo at {repo_path}")
else:
    subprocess.run([
        "git", "clone", "https://github.com/Purple-Vanguard/purple-vanguard-scenarios", str(repo_path)
    ], check=True)

os.chdir(repo_path)

subprocess.run(["python", "scripts/export_badblueprint_submission.py"], check=True)
subprocess.run([
    "python", "scripts/validate_submission_bundle.py",
    "submissions/purple_vanguard/badblueprint"
], check=True)

## F. Install vendored harness

In [ ]:
import subprocess
import sys
from pathlib import Path
import tomllib

subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", "vendor/agentbeats-lambda"
], check=True)

pyproject = Path("vendor/agentbeats-lambda/pyproject.toml")
cli_name = None
if pyproject.exists():
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = next(iter(scripts.keys()))

if not cli_name:
    raise RuntimeError("Failed to detect harness CLI entrypoint.")

print(f"Detected CLI: {cli_name}")

## G. Configure harness to use the local endpoint

In [ ]:
import os
import re
import subprocess

rg_cmd = [
    "rg",
    "--no-heading",
    "--line-number",
    "OPENAI|BASE_URL|API_BASE|endpoint|/v1/models|/v1/chat/completions",
    "vendor/agentbeats-lambda",
]
rg = subprocess.run(rg_cmd, capture_output=True, text=True, check=False)

matches = rg.stdout.splitlines()
print("Ripgrep matches:")
for line in matches:
    print(line)

env_var_candidates = set()
for line in matches:
    env_var_candidates.update(re.findall(r"[A-Z][A-Z0-9_]{2,}", line))

base_vars = sorted(v for v in env_var_candidates if "BASE_URL" in v or "API_BASE" in v or "ENDPOINT" in v)
key_vars = sorted(v for v in env_var_candidates if "API_KEY" in v or v.endswith("_KEY"))

MODEL_ENDPOINT = "http://127.0.0.1:8000/v1"

for var in base_vars:
    os.environ[var] = MODEL_ENDPOINT

for var in key_vars:
    if not os.environ.get(var):
        os.environ[var] = "local-key"

print(f"Base URL vars set: {base_vars}")
print(f"API key vars set (redacted): {key_vars}")

## H. Run FULL SCORING

In [ ]:
import os
import re
import subprocess
from pathlib import Path

results_dir = Path("results/badblueprint")
results_dir.mkdir(parents=True, exist_ok=True)

log_path = results_dir / "full_score.log"
scenario_path = "submissions/purple_vanguard/badblueprint/scenario_badblueprint.toml"

cli_cmd = globals().get("cli_name", "agentbeats-run")

model_name = globals().get("MODEL_NAME", "openai/gpt-oss-20b")
scenario_file = Path(scenario_path)
scenario_text = scenario_file.read_text()
if "--model" in scenario_text:
    scenario_text = re.sub(r"--model\s+\S+", f"--model {model_name}", scenario_text)
    scenario_file.write_text(scenario_text)

with log_path.open("w") as log_file:
    proc = subprocess.run(
        [cli_cmd, scenario_path],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

score_exit_code = proc.returncode
print(f"Full scoring exit code: {score_exit_code}")

In [ ]:
import shutil
from pathlib import Path

results_dir = Path("results/badblueprint")
agent_cards = list(Path(".").glob("**/agent-card-*.json"))

role_map = {
    "green": "agent-card-green.json",
    "attacker": "agent-card-attacker.json",
    "defender": "agent-card-defender.json",
}

for card in agent_cards:
    lower = card.name.lower()
    for role, target_name in role_map.items():
        if role in lower:
            shutil.copy2(card, results_dir / target_name)

## I. Write score_status.json

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

results_dir = Path("results/badblueprint")
commit_pin = Path("vendor/agentbeats-lambda/COMMIT_PIN.txt").read_text().strip()

success = ("score_exit_code" in globals() and score_exit_code == 0)
notes = ""
if not success:
    notes = "Scoring failed. Check full_score.log for details."

status = {
    "mode": "full",
    "ran_at": datetime.now(timezone.utc).isoformat(),
    "submission_path": "submissions/purple_vanguard/badblueprint",
    "harness_commit_pin": commit_pin,
    "model_endpoint": "http://127.0.0.1:8000/v1",
    "success": success,
    "notes": notes,
}

status_path = results_dir / "score_status.json"
status_path.write_text(json.dumps(status, indent=2, sort_keys=True))
print(f"Wrote {status_path}")

## J. Package results for download

In [ ]:
import tarfile
from pathlib import Path

results_dir = Path("results/badblueprint")
output_path = Path("results_badblueprint_colab.tgz")

with tarfile.open(output_path, "w:gz") as tar:
    tar.add(results_dir, arcname=results_dir.name)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Wrote {output_path} ({size_mb:.2f} MB)")

In [ ]:
from pathlib import Path

if Path("/content/drive").exists():
    print("Drive detected. You can copy results_badblueprint_colab.tgz to Drive if desired.")
else:
    print("Drive not mounted.")

## Cleanup: stop the model server

In [ ]:
import os
import signal
import time

if "_model_server_proc" in globals() and _model_server_proc.poll() is None:
    os.killpg(_model_server_proc.pid, signal.SIGTERM)
    time.sleep(2)
    if _model_server_proc.poll() is None:
        os.killpg(_model_server_proc.pid, signal.SIGKILL)
    print("Server stopped.")
else:
    print("Server not running.")